# 04 · Le format JSON

Une application partenaire veut recevoir nos courses de taxi en JSON, le format des API web.
On compare JSON, JSON Lines et CSV, puis on apprend à mettre à plat un JSON imbriqué.

## 1. Un échantillon de courses

On garde les 100 000 premières courses de mars, pour aller vite.

In [1]:
import json
from pathlib import Path
import pandas as pd

DATA = Path("../data")
OUTPUT = Path("../output")
OUTPUT.mkdir(exist_ok=True)

echantillon = pd.read_parquet(DATA / "yellow_tripdata_2026-03.parquet").head(100_000)
echantillon.shape

(100000, 20)

## 2. Écrire en CSV, JSON et JSON Lines

`orient="records"` écrit un objet par course ; `lines=True` met un objet par ligne : c'est le **JSON Lines**.
`date_format="iso"` écrit les dates en texte lisible (`"2026-03-01T00:02:26.000"`) au lieu d'un nombre de millisecondes.

In [2]:
chemin_csv = OUTPUT / "courses_echantillon.csv"
chemin_json = OUTPUT / "courses_echantillon.json"
chemin_jsonl = OUTPUT / "courses_echantillon.jsonl"

echantillon.to_csv(chemin_csv, index=False)
echantillon.to_json(chemin_json, orient="records", date_format="iso")
echantillon.to_json(chemin_jsonl, orient="records", lines=True, date_format="iso")

Comparons les tailles des trois fichiers.

In [3]:
def taille_mo(chemin):
    """Taille d'un fichier en Mo."""
    return round(chemin.stat().st_size / 1_000_000, 1)

pd.DataFrame({
    "format": ["CSV", "JSON", "JSON Lines"],
    "taille_mo": [taille_mo(chemin_csv), taille_mo(chemin_json), taille_mo(chemin_jsonl)],
})

,format,taille_mo
0,CSV,11.1
1,JSON,45.5
2,JSON Lines,45.5


Le JSON pèse environ **4 fois** le CSV (45,5 Mo contre 11,1 Mo).
JSON et JSON Lines ont la même taille : seule la façon de ranger les objets change.

## 3. Regarder dans les fichiers

Ce sont des fichiers texte : lisons le début du JSON classique.

In [4]:
with open(chemin_json, encoding="utf-8") as f:
    print(f.read(300))

[{"VendorID":2,"tpep_pickup_datetime":"2026-03-01T00:02:26.000","tpep_dropoff_datetime":"2026-03-01T00:13:45.000","passenger_count":1.0,"trip_distance":2.58,"RatecodeID":1.0,"store_and_fwd_flag":"N","PULocationID":48,"DOLocationID":151,"payment_type":1,"fare_amount":14.2,"extra":1.0,"mta_tax":0.5,"t


Le fichier commence par `[` : un seul grand tableau, avec un objet `{...}` par course. Voici les 2 premières lignes du JSON Lines.

In [5]:
with open(chemin_jsonl, encoding="utf-8") as f:
    print(f.readline(), end="")
    print(f.readline(), end="")

{"VendorID":2,"tpep_pickup_datetime":"2026-03-01T00:02:26.000","tpep_dropoff_datetime":"2026-03-01T00:13:45.000","passenger_count":1.0,"trip_distance":2.58,"RatecodeID":1.0,"store_and_fwd_flag":"N","PULocationID":48,"DOLocationID":151,"payment_type":1,"fare_amount":14.2,"extra":1.0,"mta_tax":0.5,"tip_amount":4.99,"tolls_amount":0.0,"improvement_surcharge":1.0,"total_amount":24.94,"congestion_surcharge":2.5,"Airport_fee":0.0,"cbd_congestion_fee":0.75}
{"VendorID":2,"tpep_pickup_datetime":"2026-03-01T00:19:33.000","tpep_dropoff_datetime":"2026-03-01T00:28:21.000","passenger_count":1.0,"trip_distance":1.5,"RatecodeID":1.0,"store_and_fwd_flag":"N","PULocationID":238,"DOLocationID":166,"payment_type":1,"fare_amount":10.0,"extra":1.0,"mta_tax":0.5,"tip_amount":0.0,"tolls_amount":0.0,"improvement_surcharge":1.0,"total_amount":15.0,"congestion_surcharge":2.5,"Airport_fee":0.0,"cbd_congestion_fee":0.0}


En JSON Lines, chaque ligne est un objet complet : pas de crochet autour, pas de virgule entre les lignes.
Les noms de colonnes (`"VendorID"`, `"tpep_pickup_datetime"`…) sont répétés pour **chaque** course : c'est ce qui rend JSON si lourd.

## 4. Relire : plus lent, et les types se perdent

On relit le CSV puis le JSON Lines en mesurant la durée avec `%%time` (*Wall time* = durée réelle écoulée).

In [6]:
%%time
relu_csv = pd.read_csv(chemin_csv)

CPU times: user 199 ms, sys: 17.1 ms, total: 216 ms
Wall time: 216 ms


In [7]:
%%time
relu_json = pd.read_json(chemin_jsonl, lines=True)

CPU times: user 819 ms, sys: 172 ms, total: 991 ms
Wall time: 990 ms


Le JSON Lines se relit plusieurs fois plus lentement que le CSV : il y a 4 fois plus de texte à analyser.

In [8]:
pd.DataFrame({"parquet": echantillon.dtypes, "csv": relu_csv.dtypes, "json": relu_json.dtypes}).head(5)

,parquet,csv,json
VendorID,int32,int64,int64
tpep_pickup_datetime,datetime64[us],str,str
tpep_dropoff_datetime,datetime64[us],str,str
passenger_count,float64,float64,int64
trip_distance,float64,float64,float64


JSON ne connaît que six types (objet, tableau, chaîne, nombre, booléen, `null`), pas de date : comme en CSV, les dates reviennent en **texte** (`str`).
Pour les nombres, pandas devine : `VendorID` passe de int32 à int64 et, en JSON, `passenger_count` devient même un entier.

On reconvertit les dates avec `pd.to_datetime`.

In [9]:
relu_json["tpep_pickup_datetime"] = pd.to_datetime(relu_json["tpep_pickup_datetime"])
relu_json["tpep_dropoff_datetime"] = pd.to_datetime(relu_json["tpep_dropoff_datetime"])
relu_json.dtypes.head(3)

VendorID                          int64
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
dtype: object

Les deux colonnes sont redevenues des dates. Comme en CSV, cette conversion est à refaire à chaque lecture.

## 5. Lire par paquets

Un JSON Lines trop gros pour la mémoire se lit par paquets (*chunks*) avec `chunksize`.

In [10]:
for paquet in pd.read_json(chemin_jsonl, lines=True, chunksize=25_000):
    print(len(paquet), "courses, montant total :", round(paquet["total_amount"].sum()), "$")

25000 courses, montant total : 665634 $


25000 courses, montant total : 718874 $


25000 courses, montant total : 800722 $


25000 courses, montant total : 759619 $


4 paquets de 25 000 courses : pandas ne charge que 25 000 lignes à la fois.
Impossible avec le JSON classique, qu'il faut charger en entier avant de lire le premier objet.

## 6. Les accents : `force_ascii`

Par défaut, pandas remplace chaque lettre accentuée par un code (`\u00e9` pour « é »). `force_ascii=False` garde le texte lisible.

In [11]:
quartiers = pd.DataFrame({"quartier": ["Gbégamey", "Cadjèhoun"], "ville": ["Cotonou", "Cotonou"]})
print(quartiers.to_json(orient="records"))
print(quartiers.to_json(orient="records", force_ascii=False))

[{"quartier":"Gb\u00e9gamey","ville":"Cotonou"},{"quartier":"Cadj\u00e8houn","ville":"Cotonou"}]
[{"quartier":"Gbégamey","ville":"Cotonou"},{"quartier":"Cadjèhoun","ville":"Cotonou"}]


Les deux lignes sont du JSON valide et se relisent de la même façon, mais seule la seconde est lisible par un humain.

## 7. Un JSON imbriqué : les commandes de la boutique

Une API renvoie souvent des objets dans des objets. Voici 3 commandes de la boutique : payée en deux fois, en une fois, pas encore payée.

In [12]:
commandes = [
    {"commande_id": "CMD0000001", "date_commande": "2026-10-01T07:42:10Z",
     "client": {"id": "CL000042", "telephone": "0197000042"},
     "livraison": {"quartier": "Gbégamey", "ville": "Cotonou"},
     "paiements": [{"moyen": "Mobile Money", "montant_fcfa": 20000},
                   {"moyen": "Paiement à la livraison", "montant_fcfa": 25000}]},
    {"commande_id": "CMD0000002", "date_commande": "2026-10-01T09:15:00Z",
     "client": {"id": "CL000107", "telephone": "0196000107"},
     "livraison": {"quartier": "Godomey", "ville": "Abomey-Calavi"},
     "paiements": [{"moyen": "Mobile Money", "montant_fcfa": 12500}]},
    {"commande_id": "CMD0000003", "date_commande": "2026-10-02T18:03:45Z",
     "client": {"id": "CL000233", "telephone": "0195000233"},
     "livraison": {"quartier": "Cadjèhoun", "ville": "Cotonou"},
     "paiements": []},
]

On l'enregistre avec le module `json` de Python (`indent=2` pour un fichier aéré, `ensure_ascii=False` pour les accents), puis on le relit.

In [13]:
chemin_commandes = OUTPUT / "commandes.json"
with open(chemin_commandes, "w", encoding="utf-8") as f:
    json.dump(commandes, f, ensure_ascii=False, indent=2)

with open(chemin_commandes, encoding="utf-8") as f:
    commandes_relues = json.load(f)
commandes_relues[0]

{'commande_id': 'CMD0000001',
 'date_commande': '2026-10-01T07:42:10Z',
 'client': {'id': 'CL000042', 'telephone': '0197000042'},
 'livraison': {'quartier': 'Gbégamey', 'ville': 'Cotonou'},
 'paiements': [{'moyen': 'Mobile Money', 'montant_fcfa': 20000},
  {'moyen': 'Paiement à la livraison', 'montant_fcfa': 25000}]}

Les objets JSON redeviennent des dictionnaires Python (`client`, `livraison`) et les tableaux des listes (`paiements`).

## 8. Mettre à plat : une ligne par commande

Un tableau est plat, un JSON est un arbre : aplatir, c'est choisir ce qu'est **une ligne**. `pd.json_normalize` transforme les objets en colonnes (`client.id` devient `client_id`).

In [14]:
plat = pd.json_normalize(commandes_relues, sep="_")
plat

,commande_id,date_commande,paiements,client_id,client_telephone,livraison_quartier,livraison_ville
0,CMD0000001,2026-10-01T07:42:10Z,"[{'moyen': 'Mobile Money', 'montant_fcfa': 200...",CL000042,0197000042,Gbégamey,Cotonou
1,CMD0000002,2026-10-01T09:15:00Z,"[{'moyen': 'Mobile Money', 'montant_fcfa': 125...",CL000107,0196000107,Godomey,Abomey-Calavi
2,CMD0000003,2026-10-02T18:03:45Z,[],CL000233,0195000233,Cadjèhoun,Cotonou


3 lignes, une par commande. La liste `paiements` reste entière dans une seule case : `json_normalize` transforme les objets en colonnes, pas les listes.

## 9. Une ligne par paiement

Pour analyser les paiements, `record_path` transforme la liste en lignes et `meta` garde le contexte de la commande.

In [15]:
paiements = pd.json_normalize(commandes_relues, record_path="paiements",
                              meta=["commande_id", ["client", "id"]], sep="_")
paiements

,moyen,montant_fcfa,commande_id,client_id
0,Mobile Money,20000,CMD0000001,CL000042
1,Paiement à la livraison,25000,CMD0000001,CL000042
2,Mobile Money,12500,CMD0000002,CL000107


3 lignes : deux paiements pour la première commande, un pour la deuxième, aucun pour la troisième (pas encore payée, elle disparaît).
Réponse à l'exercice de la slide 14 : 3 lignes dans `plat`, puis 3 dans `paiements`.

## Exercices

### ✍️ Exercice 1 — Ajouter une course

Écrivez les 3 premières courses de l'échantillon dans `output/exo04_ajout.jsonl`, puis ajoutez-y la dernière course **sans réécrire le fichier** (`mode="a"`). Relisez-le.
Pourquoi est-ce impossible proprement avec le JSON classique ?

In [16]:
# ✅ Solution
chemin_exo1 = OUTPUT / "exo04_ajout.jsonl"
echantillon.head(3).to_json(chemin_exo1, orient="records", lines=True, date_format="iso")
echantillon.tail(1).to_json(chemin_exo1, orient="records", lines=True, date_format="iso", mode="a")
pd.read_json(chemin_exo1, lines=True)[["tpep_pickup_datetime", "total_amount"]]

,tpep_pickup_datetime,total_amount
0,2026-03-01T00:02:26.000,24.94
1,2026-03-01T00:19:33.000,15.00
2,2026-03-01T00:07:20.000,17.22
3,2026-03-02T11:31:24.000,9.24


4 courses : la nouvelle a simplement été écrite à la fin du fichier.
En JSON classique, tout est dans un seul tableau `[ ... ]` : pour ajouter une course, il faut tout relire et tout réécrire.

### ✍️ Exercice 2 — Un JSON lisible

Comptez les courses de l'échantillon par type de paiement (0 = non renseigné, 1 = carte, 2 = espèces, 3 = gratuit, 4 = litige) et exportez le résultat en JSON lisible (`indent=2`, `force_ascii=False`) dans `output/exo04_paiements.json`.

In [17]:
# ✅ Solution
noms = {0: "non renseigné", 1: "carte", 2: "espèces", 3: "gratuit", 4: "litige"}
par_paiement = echantillon["payment_type"].map(noms).value_counts()

chemin_exo2 = OUTPUT / "exo04_paiements.json"
par_paiement.to_json(chemin_exo2, indent=2, force_ascii=False)
print(chemin_exo2.read_text(encoding="utf-8"))

{
  "carte":85946,
  "espèces":12080,
  "litige":1435,
  "gratuit":539
}


Environ 86 % des courses sont payées par carte (le code 0 est absent de l'échantillon).
Le mot « espèces » reste lisible grâce à `force_ascii=False`.

## À retenir

- JSON : du texte lisible et imbriqué, la langue des API web.
- Lourd (environ 4 fois le CSV) et lent : les noms de colonnes sont répétés pour chaque course.
- Pas de type date : après `read_json`, on reconvertit avec `pd.to_datetime`.
- JSON Lines (un objet par ligne) se lit par paquets et accepte les ajouts.
- `pd.json_normalize` met à plat : les objets deviennent des colonnes, une liste devient des lignes (`record_path`).